In [1]:
# W7D2 — Haystack Retrieval: BM25 & Dense Retrieval

!pip install -q haystack-ai pypdf reportlab sentence-transformers sentence-transformers-haystack

print("✅ Required packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.7/665.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.1 MB/s eta 0:00:00
✅ Required packages installed!


In [2]:
from reportlab.pdfgen import canvas
from pathlib import Path

from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [3]:
pdf_folder = Path("w7d2_pdfs")
pdf_folder.mkdir(exist_ok=True)

pdf_data = {
    "python.pdf": (
        "Python Programming",
        "Python is a high-level programming language used for web development, "
        "automation, data science, artificial intelligence, and machine learning."
    ),
    "machine_learning.pdf": (
        "Machine Learning",
        "Machine learning is a branch of artificial intelligence where computers "
        "learn patterns from data and use those patterns to make predictions."
    ),
    "database.pdf": (
        "Database Systems",
        "A database is an organized collection of data. SQL databases store "
        "structured information using tables, rows, and columns."
    ),
    "rag.pdf": (
        "Retrieval Augmented Generation",
        "RAG combines information retrieval with a language model. Relevant "
        "documents are retrieved first and then provided to the language model "
        "as context."
    ),
    "haystack.pdf": (
        "Haystack Framework",
        "Haystack is an open-source framework for building search, question "
        "answering, and retrieval augmented generation applications."
    )
}

for filename, (title, content) in pdf_data.items():
    file_path = pdf_folder / filename

    pdf = canvas.Canvas(str(file_path))
    pdf.setTitle(title)

    pdf.drawString(50, 800, title)

    # Write content across multiple lines
    text = pdf.beginText(50, 760)
    text.setFont("Helvetica", 11)

    words = content.split()
    line = ""

    for word in words:
        if len(line) + len(word) > 80:
            text.textLine(line)
            line = word
        else:
            line += " " + word

    if line:
        text.textLine(line)

    pdf.drawText(text)
    pdf.save()

print("✅ 5 PDF documents created!")

for file in sorted(pdf_folder.glob("*.pdf")):
    print("📄", file)

✅ 5 PDF documents created!
📄 w7d2_pdfs/database.pdf
📄 w7d2_pdfs/haystack.pdf
📄 w7d2_pdfs/machine_learning.pdf
📄 w7d2_pdfs/python.pdf
📄 w7d2_pdfs/rag.pdf


In [4]:
from pypdf import PdfReader

pdf_documents = []

for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    reader = PdfReader(str(pdf_file))

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    pdf_documents.append(
        Document(
            content=text.strip(),
            meta={
                "source": pdf_file.name
            }
        )
    )

print("✅ PDF text extracted successfully!")
print("Total PDF documents:", len(pdf_documents))

for i, doc in enumerate(pdf_documents, 1):
    print(f"\n{i}. {doc.meta['source']}")
    print("   Text:", doc.content[:150], "...")

✅ PDF text extracted successfully!
Total PDF documents: 5

1. database.pdf
   Text: Database Systems
 A database is an organized collection of data. SQL databases store structured
information using tables, rows, and columns. ...

2. haystack.pdf
   Text: Haystack Framework
 Haystack is an open-source framework for building search, question answering,
and retrieval augmented generation applications. ...

3. machine_learning.pdf
   Text: Machine Learning
 Machine learning is a branch of artificial intelligence where computers learn
patterns from data and use those patterns to make pred ...

4. python.pdf
   Text: Python Programming
 Python is a high-level programming language used for web development,
automation, data science, artificial intelligence, and machi ...

5. rag.pdf
   Text: Retrieval Augmented Generation
 RAG combines information retrieval with a language model. Relevant documents are
retrieved first and then provided to  ...


In [5]:
# Create Haystack DocumentStore for PDF documents

pdf_document_store = InMemoryDocumentStore()

pdf_document_store.write_documents(pdf_documents)

print("✅ PDF documents indexed successfully!")
print("Total documents:", pdf_document_store.count_documents())

✅ PDF documents indexed successfully!
Total documents: 5


In [6]:
bm25_retriever = InMemoryBM25Retriever(
    document_store=pdf_document_store
)

print("✅ BM25 Retriever created successfully!")

✅ BM25 Retriever created successfully!


In [7]:
bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    "bm25_retriever",
    bm25_retriever
)

print("✅ W7D2 BM25 pipeline created successfully!")
print("Pipeline: Query → BM25 Retriever → PDF Documents")

✅ W7D2 BM25 pipeline created successfully!
Pipeline: Query → BM25 Retriever → PDF Documents


In [8]:
evaluation_questions = [
    "What is Python used for?",
    "What is machine learning?",
    "How does machine learning make predictions?",
    "What is a database?",
    "How does an SQL database organize data?",
    "What is RAG?",
    "How does RAG provide context to a language model?",
    "What is Haystack?",
    "What type of applications can Haystack build?",
    "What technology combines retrieval with a language model?"
]

expected_topics = [
    "python.pdf",
    "machine_learning.pdf",
    "machine_learning.pdf",
    "database.pdf",
    "database.pdf",
    "rag.pdf",
    "rag.pdf",
    "haystack.pdf",
    "haystack.pdf",
    "rag.pdf"
]

print("✅ 10 evaluation questions created!")
print()

for i, question in enumerate(evaluation_questions, 1):
    print(f"Q{i}. {question}")

✅ 10 evaluation questions created!

Q1. What is Python used for?
Q2. What is machine learning?
Q3. How does machine learning make predictions?
Q4. What is a database?
Q5. How does an SQL database organize data?
Q6. What is RAG?
Q7. How does RAG provide context to a language model?
Q8. What is Haystack?
Q9. What type of applications can Haystack build?
Q10. What technology combines retrieval with a language model?


In [9]:
bm25_results = []

print("=" * 80)
print("W7D2 — BM25 RETRIEVAL RESULTS")
print("=" * 80)

for i, question in enumerate(evaluation_questions, 1):

    result = bm25_pipeline.run({
        "bm25_retriever": {
            "query": question,
            "top_k": 1
        }
    })

    docs = result["bm25_retriever"]["documents"]

    if docs:
        top_doc = docs[0]

        bm25_results.append({
            "question": question,
            "retrieved": top_doc.meta["source"],
            "score": top_doc.score
        })

        print(f"\nQ{i}. {question}")
        print("Retrieved PDF:", top_doc.meta["source"])
        print("Score:", top_doc.score)

    else:
        bm25_results.append({
            "question": question,
            "retrieved": None,
            "score": None
        })

        print(f"\nQ{i}. {question}")
        print("No document retrieved.")

print("\n" + "=" * 80)
print("✅ BM25 completed for all 10 questions")

W7D2 — BM25 RETRIEVAL RESULTS

Q1. What is Python used for?
Retrieved PDF: python.pdf
Score: 5.377097515668969

Q2. What is machine learning?
Retrieved PDF: machine_learning.pdf
Score: 3.044194781133945

Q3. How does machine learning make predictions?
Retrieved PDF: machine_learning.pdf
Score: 6.094361727382729

Q4. What is a database?
Retrieved PDF: database.pdf
Score: 2.930136427072574

Q5. How does an SQL database organize data?
Retrieved PDF: database.pdf
Score: 5.758915069166558

Q6. What is RAG?
Retrieved PDF: rag.pdf
Score: 1.8454459209813696

Q7. How does RAG provide context to a language model?
Retrieved PDF: rag.pdf
Score: 8.137003711467171

Q8. What is Haystack?
Retrieved PDF: haystack.pdf
Score: 2.6235415132097373

Q9. What type of applications can Haystack build?
Retrieved PDF: haystack.pdf
Score: 4.6032274012215115

Q10. What technology combines retrieval with a language model?
Retrieved PDF: rag.pdf
Score: 8.404336417157596

✅ BM25 completed for all 10 questions


In [10]:
bm25_correct = 0

print("=" * 80)
print("W7D2 — BM25 MANUAL EVALUATION")
print("=" * 80)

for i, result in enumerate(bm25_results):

    expected = expected_topics[i]
    retrieved = result["retrieved"]

    correct = retrieved == expected

    if correct:
        bm25_correct += 1

    print(f"\nQ{i+1}: {result['question']}")
    print("Expected :", expected)
    print("Retrieved:", retrieved)
    print("Result   :", "✅ Correct" if correct else "❌ Incorrect")

bm25_precision = bm25_correct / len(evaluation_questions)

print("\n" + "=" * 80)
print(f"Correct Retrievals : {bm25_correct}/{len(evaluation_questions)}")
print(f"BM25 Precision     : {bm25_precision:.2f}")
print("=" * 80)

W7D2 — BM25 MANUAL EVALUATION

Q1: What is Python used for?
Expected : python.pdf
Retrieved: python.pdf
Result   : ✅ Correct

Q2: What is machine learning?
Expected : machine_learning.pdf
Retrieved: machine_learning.pdf
Result   : ✅ Correct

Q3: How does machine learning make predictions?
Expected : machine_learning.pdf
Retrieved: machine_learning.pdf
Result   : ✅ Correct

Q4: What is a database?
Expected : database.pdf
Retrieved: database.pdf
Result   : ✅ Correct

Q5: How does an SQL database organize data?
Expected : database.pdf
Retrieved: database.pdf
Result   : ✅ Correct

Q6: What is RAG?
Expected : rag.pdf
Retrieved: rag.pdf
Result   : ✅ Correct

Q7: How does RAG provide context to a language model?
Expected : rag.pdf
Retrieved: rag.pdf
Result   : ✅ Correct

Q8: What is Haystack?
Expected : haystack.pdf
Retrieved: haystack.pdf
Result   : ✅ Correct

Q9: What type of applications can Haystack build?
Expected : haystack.pdf
Retrieved: haystack.pdf
Result   : ✅ Correct

Q10: What tec

In [11]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder
)

from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("✅ Dense retrieval components imported!")

✅ Dense retrieval components imported!


In [12]:
document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

document_embedder.warm_up()

print("✅ Document embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Document embedding model loaded!


In [13]:
embedded_result = document_embedder.run(
    documents=pdf_documents
)

embedded_documents = embedded_result["documents"]

print("✅ PDF document embeddings generated!")
print("Number of documents:", len(embedded_documents))
print("Embedding dimension:", len(embedded_documents[0].embedding))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ PDF document embeddings generated!
Number of documents: 5
Embedding dimension: 384


In [14]:
dense_document_store = InMemoryDocumentStore()

dense_document_store.write_documents(embedded_documents)

print("✅ Dense document store created!")
print("Total documents:", dense_document_store.count_documents())

✅ Dense document store created!
Total documents: 5


In [15]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=dense_document_store
)

print("✅ Dense Retriever created successfully!")

✅ Dense Retriever created successfully!


In [16]:
text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

text_embedder.warm_up()

print("✅ Text embedding model loaded successfully!")

✅ Text embedding model loaded successfully!


In [17]:
dense_pipeline = Pipeline()

dense_pipeline.add_component(
    "text_embedder",
    text_embedder
)

dense_pipeline.add_component(
    "dense_retriever",
    dense_retriever
)

dense_pipeline.connect(
    "text_embedder.embedding",
    "dense_retriever.query_embedding"
)

print("✅ Dense retrieval pipeline created!")
print("Pipeline: Question → Text Embedder → Dense Retriever → PDF Documents")

✅ Dense retrieval pipeline created!
Pipeline: Question → Text Embedder → Dense Retriever → PDF Documents


In [18]:
test_question = "What is machine learning?"

dense_test_result = dense_pipeline.run(
    {
        "text_embedder": {
            "text": test_question
        },
        "dense_retriever": {
            "top_k": 1
        }
    }
)

top_doc = dense_test_result["dense_retriever"]["documents"][0]

print("Question:", test_question)
print("Retrieved PDF:", top_doc.meta["source"])
print("Similarity Score:", top_doc.score)
print("Content:", top_doc.content)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: What is machine learning?
Retrieved PDF: machine_learning.pdf
Similarity Score: 0.8333791538419251
Content: Machine Learning
 Machine learning is a branch of artificial intelligence where computers learn
patterns from data and use those patterns to make predictions.


In [19]:
dense_results = []

for i, question in enumerate(evaluation_questions, 1):
    result = dense_pipeline.run(
        {
            "text_embedder": {
                "text": question
            },
            "dense_retriever": {
                "top_k": 1
            }
        }
    )

    top_doc = result["dense_retriever"]["documents"][0]

    dense_results.append({
        "question": question,
        "retrieved": top_doc.meta["source"],
        "score": top_doc.score
    })

    print(f"Q{i}. {question}")
    print(f"   Retrieved: {top_doc.meta['source']}")
    print(f"   Score: {top_doc.score:.4f}")
    print()

print("✅ Dense retrieval completed for all 10 questions!")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q1. What is Python used for?
   Retrieved: python.pdf
   Score: 0.8048



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q2. What is machine learning?
   Retrieved: machine_learning.pdf
   Score: 0.8334



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Q8. What is Haystack?
   Retrieved: haystack.pdf
   Score: 0.4514



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q9. What type of applications can Haystack build?
   Retrieved: haystack.pdf
   Score: 0.5407



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q10. What technology combines retrieval with a language model?
   Retrieved: rag.pdf
   Score: 0.6880

✅ Dense retrieval completed for all 10 questions!


In [20]:
dense_correct = 0

print("📊 Dense Retrieval Evaluation")
print("=" * 60)

for i, result in enumerate(dense_results):
    expected = expected_topics[i]
    retrieved = result["retrieved"]

    correct = retrieved == expected

    if correct:
        dense_correct += 1

    status = "✅ Correct" if correct else "❌ Wrong"

    print(f"Q{i+1}: {status}")
    print(f"   Expected : {expected}")
    print(f"   Retrieved: {retrieved}")
    print()

dense_precision = dense_correct / len(evaluation_questions)

print("=" * 60)
print(f"Correct answers: {dense_correct}/{len(evaluation_questions)}")
print(f"Dense Retrieval Precision: {dense_precision:.2%}")

📊 Dense Retrieval Evaluation
Q1: ✅ Correct
   Expected : python.pdf
   Retrieved: python.pdf

Q2: ✅ Correct
   Expected : machine_learning.pdf
   Retrieved: machine_learning.pdf

Q3: ✅ Correct
   Expected : machine_learning.pdf
   Retrieved: machine_learning.pdf

Q4: ✅ Correct
   Expected : database.pdf
   Retrieved: database.pdf

Q5: ✅ Correct
   Expected : database.pdf
   Retrieved: database.pdf

Q6: ✅ Correct
   Expected : rag.pdf
   Retrieved: rag.pdf

Q7: ✅ Correct
   Expected : rag.pdf
   Retrieved: rag.pdf

Q8: ✅ Correct
   Expected : haystack.pdf
   Retrieved: haystack.pdf

Q9: ✅ Correct
   Expected : haystack.pdf
   Retrieved: haystack.pdf

Q10: ✅ Correct
   Expected : rag.pdf
   Retrieved: rag.pdf

Correct answers: 10/10
Dense Retrieval Precision: 100.00%


In [21]:
print("📊 BM25 vs Dense Retrieval Comparison")
print("=" * 70)

print(f"BM25 Precision   : {bm25_precision:.2%}")
print(f"Dense Precision  : {dense_precision:.2%}")

print()
print("Question-wise comparison:")
print("-" * 70)

for i in range(len(evaluation_questions)):
    bm25_doc = bm25_results[i]["retrieved"]
    dense_doc = dense_results[i]["retrieved"]
    expected = expected_topics[i]

    bm25_status = "✅" if bm25_doc == expected else "❌"
    dense_status = "✅" if dense_doc == expected else "❌"

    print(f"Q{i+1}. {evaluation_questions[i]}")
    print(f"   Expected: {expected}")
    print(f"   BM25   : {bm25_status} {bm25_doc}")
    print(f"   Dense  : {dense_status} {dense_doc}")
    print()

print("=" * 70)
print("✅ BM25 and Dense retrieval comparison completed!")

📊 BM25 vs Dense Retrieval Comparison
BM25 Precision   : 100.00%
Dense Precision  : 100.00%

Question-wise comparison:
----------------------------------------------------------------------
Q1. What is Python used for?
   Expected: python.pdf
   BM25   : ✅ python.pdf
   Dense  : ✅ python.pdf

Q2. What is machine learning?
   Expected: machine_learning.pdf
   BM25   : ✅ machine_learning.pdf
   Dense  : ✅ machine_learning.pdf

Q3. How does machine learning make predictions?
   Expected: machine_learning.pdf
   BM25   : ✅ machine_learning.pdf
   Dense  : ✅ machine_learning.pdf

Q4. What is a database?
   Expected: database.pdf
   BM25   : ✅ database.pdf
   Dense  : ✅ database.pdf

Q5. How does an SQL database organize data?
   Expected: database.pdf
   BM25   : ✅ database.pdf
   Dense  : ✅ database.pdf

Q6. What is RAG?
   Expected: rag.pdf
   BM25   : ✅ rag.pdf
   Dense  : ✅ rag.pdf

Q7. How does RAG provide context to a language model?
   Expected: rag.pdf
   BM25   : ✅ rag.pdf
   Dense 

In [22]:
print("📝 W7D2 SELF-REVIEW")
print("=" * 60)

print("\n1. What I built:")
print("- Indexed 5 PDF documents using Haystack.")
print("- Built a BM25 retrieval pipeline.")
print("- Built a Dense retrieval pipeline using Sentence Transformers.")
print("- Tested both methods with the same 10 questions.")
print("- Compared retrieval precision manually.")

print("\n2. Design decision:")
print("- BM25 was used for keyword-based retrieval.")
print("- Dense retrieval was used for semantic similarity.")
print("- The same evaluation questions were used for a fair comparison.")

print("\n3. Challenge and solution:")
print("- Challenge: Setting up dense retrieval components correctly.")
print("- Solution: Used Haystack's Sentence Transformers integrations")
print("  and connected text embeddings to the dense retriever.")

print("\n4. Improvement I would make:")
print("- Use a larger and more diverse PDF dataset.")
print("- Evaluate Top-K retrieval instead of only Top-1.")
print("- Add automated evaluation metrics.")

print("\n5. Final Results:")
print(f"- BM25 Precision  : {bm25_precision:.2%}")
print(f"- Dense Precision : {dense_precision:.2%}")

print("\n✅ W7D2 practical completed!")

📝 W7D2 SELF-REVIEW

1. What I built:
- Indexed 5 PDF documents using Haystack.
- Built a BM25 retrieval pipeline.
- Built a Dense retrieval pipeline using Sentence Transformers.
- Tested both methods with the same 10 questions.
- Compared retrieval precision manually.

2. Design decision:
- BM25 was used for keyword-based retrieval.
- Dense retrieval was used for semantic similarity.
- The same evaluation questions were used for a fair comparison.

3. Challenge and solution:
- Challenge: Setting up dense retrieval components correctly.
- Solution: Used Haystack's Sentence Transformers integrations
  and connected text embeddings to the dense retriever.

4. Improvement I would make:
- Use a larger and more diverse PDF dataset.
- Evaluate Top-K retrieval instead of only Top-1.
- Add automated evaluation metrics.

5. Final Results:
- BM25 Precision  : 100.00%
- Dense Precision : 100.00%

✅ W7D2 practical completed!
